In [1]:
import os
import json
from pathlib import Path

import cv2
import timm
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    precision_recall_fscore_support,
    f1_score
)
from tqdm.auto import tqdm

In [2]:
from pathlib import Path

MODEL_DIR = Path("/kaggle/input/models/mannyquin1/avatar-attribute-model/pytorch/default/1")

print("Files in model directory:\n")

for file in MODEL_DIR.iterdir():
    print(file.name)
    

Files in model directory:

avatar_attribute_model.pth


In [3]:
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

IMAGE_SIZE = 224
BATCH_SIZE = 32
NUM_WORKERS = 2

CHECKPOINT_PATH = MODEL_DIR / "avatar_attribute_model.pth"
print(CHECKPOINT_PATH)

TEST_CSV = "/kaggle/input/datasets/mannyquin1/celeba-datasets/celeba_test.csv"
IMAGE_DIR = "/kaggle/input/datasets/jessicali9530/celeba-dataset/img_align_celeba/img_align_celeba"

OUTPUT_DIR = Path("evaluation_results")
OUTPUT_DIR.mkdir(exist_ok=True)

/kaggle/input/models/mannyquin1/avatar-attribute-model/pytorch/default/1/avatar_attribute_model.pth


In [4]:
HAIR_COLOR_CLASSES = [
    "Black","Brown","Blond","Gray","Other"
]

HAIR_TEXTURE_CLASSES = [
    "Straight","Wavy"
]

BINARY_ATTRIBUTES = [
    "Bald","Bangs","Receding Hairline",
    "Mustache","Goatee","Sideburns",
    "Glasses","Hat","Earrings",
    "Oval Face","High Cheekbones",
    "Arched Eyebrows","Bushy Eyebrows",
    "Big Nose","Big Lips"
]


In [5]:
import albumentations as A
from albumentations.pytorch import ToTensorV2

eval_transform = A.Compose([
    A.Resize(IMAGE_SIZE, IMAGE_SIZE),

    A.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    ),

    ToTensorV2()
])

In [6]:
class CelebADataset(Dataset):

    def __init__(
        self,
        dataframe,
        image_dir,
        transform=None
    ):
        self.df = dataframe.reset_index(drop=True)
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        img_path = os.path.join(
            self.image_dir,
            row["image_id"]
        )

        image = cv2.imread(img_path)

        if image is None:
            raise FileNotFoundError(
                f"Image not found: {img_path}"
            )

        image = cv2.cvtColor(
            image,
            cv2.COLOR_BGR2RGB
        )

        if self.transform:
            image = self.transform(image=image)["image"]

        hair_color = torch.as_tensor(
            row["hair_color"],
            dtype=torch.long
        )

        texture = row["hair_texture"]

        # Unknown → ignore during training
        if texture == 2:
            texture = -1

        hair_texture = torch.as_tensor(
            texture,
            dtype=torch.long
        )

        hair_attributes = torch.as_tensor([
            row["bald"],
            row["bangs"],
            row["receding"]
        ], dtype=torch.float32)
        

        facial_hair = torch.as_tensor([
            row["mustache"],
            row["goatee"],
            row["sideburns"]
        ], dtype=torch.float32)


        accessories = torch.as_tensor([
            row["glasses"],
            row["hat"],
            row["earrings"]
        ], dtype=torch.float32)


        face_features = torch.as_tensor([
            row["oval_face"],
            row["high_cheekbones"],
            row["arched_eyebrows"],
            row["bushy_eyebrows"],
            row["big_nose"],
            row["big_lips"]
        ], dtype=torch.float32)

        return {
            "image_id": row["image_id"],

            "image": image,

            "hair_color": hair_color,

            "hair_texture": hair_texture,

            "hair_attributes": hair_attributes,

            "facial_hair": facial_hair,

            "accessories": accessories,

            "face_features": face_features

        }

In [7]:
test_df = pd.read_csv(TEST_CSV)

test_dataset = CelebADataset(
    dataframe=test_df,
    image_dir=IMAGE_DIR,
    transform=eval_transform
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

print(f"Test Samples: {len(test_dataset)}")

Test Samples: 20260


In [8]:
MODEL_NAME = "efficientnet_b0"

HIDDEN_COLOR = 256
HIDDEN_TEXTURE = 128
HIDDEN_BINARY = 512

NUM_HAIR_COLOR = 5
NUM_HAIR_TEXTURE = 2
NUM_BINARY = 15

In [9]:
class AvatarAttributeModel(nn.Module):

    def __init__(self):

        super().__init__()

        # ----------------------------
        # Backbone
        # ----------------------------

        self.backbone = timm.create_model(

            MODEL_NAME,

            pretrained=True,

            num_classes=0,

            global_pool="avg"

        )

        feature_dim = self.backbone.num_features


        # ----------------------------
        # Heads
        # ----------------------------

        self.hair_color = nn.Sequential(

            nn.Linear(feature_dim, HIDDEN_COLOR),

            nn.BatchNorm1d(HIDDEN_COLOR),

            nn.ReLU(inplace=True),

            nn.Dropout(0.3),

            nn.Linear(HIDDEN_COLOR, NUM_HAIR_COLOR)

        )

        self.hair_texture = nn.Sequential(

            nn.Linear(feature_dim,HIDDEN_TEXTURE),

            nn.BatchNorm1d(HIDDEN_TEXTURE),

            nn.ReLU(inplace=True),

            nn.Dropout(0.3),

            nn.Linear(HIDDEN_TEXTURE,NUM_HAIR_TEXTURE)

        )

        self.binary = nn.Sequential(

            nn.Linear(feature_dim,HIDDEN_BINARY),

            nn.BatchNorm1d(HIDDEN_BINARY),

            nn.ReLU(inplace=True),

            nn.Dropout(0.3),
            
            nn.Linear(HIDDEN_BINARY,NUM_BINARY)

        )

    def forward(self, x):

        features = self.backbone(x)


        return {

            "hair_color":
                self.hair_color(features),

            "hair_texture":
                self.hair_texture(features),

            "binary":
                self.binary(features)

        }

In [10]:
checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)

print(checkpoint.keys())

model = AvatarAttributeModel()
model.load_state_dict(checkpoint["model_state_dict"])
model.to(DEVICE)
model.eval()


dict_keys(['model_state_dict', 'model_name', 'image_size', 'num_hair_color', 'num_hair_texture', 'num_binary'])


model.safetensors:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

AvatarAttributeModel(
  (backbone): EfficientNet(
    (conv_stem): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    (bn1): BatchNormAct2d(
      32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True
      (drop): Identity()
      (act): SiLU(inplace=True)
    )
    (blocks): Sequential(
      (0): Sequential(
        (0): DepthwiseSeparableConv(
          (conv_dw): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
          (bn1): BatchNormAct2d(
            32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True
            (drop): Identity()
            (act): SiLU(inplace=True)
          )
          (aa): Identity()
          (se): SqueezeExcite(
            (conv_reduce): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
            (act1): SiLU(inplace=True)
            (conv_expand): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
            (gate): Sigmoid()
          )
          (co

In [11]:
import time

# ------------------------------------------------------------------
# Store all predictions and metadata
# ------------------------------------------------------------------

results = {
    "image_ids": [],

    "hair_color_targets": [],
    "hair_color_preds": [],

    "hair_texture_targets": [],
    "hair_texture_preds": [],

    "binary_targets": [],
    "binary_logits": [],
    "binary_probs": []
}

# ------------------------------------------------------------------
# Inference
# ------------------------------------------------------------------

model.eval()

start_time = time.time()

with torch.no_grad():

    with torch.autocast(
        device_type="cuda",
        enabled=(DEVICE.type == "cuda")
    ):

        for batch in tqdm(
            test_loader,
            desc="Running Inference"
        ):

            images = batch["image"].to(
                DEVICE,
                non_blocking=True
            )

            outputs = model(images)

            # ----------------------------
            # Hair Color
            # ----------------------------

            hair_color_logits = outputs["hair_color"]

            hair_color_preds = torch.argmax(
                hair_color_logits,
                dim=1
            )

            # ----------------------------
            # Hair Texture
            # ----------------------------

            hair_texture_logits = outputs["hair_texture"]

            hair_texture_preds = torch.argmax(
                hair_texture_logits,
                dim=1
            )

            # ----------------------------
            # Binary Attributes
            # ----------------------------

            binary_logits = outputs["binary"]

            binary_probs = torch.sigmoid(
                binary_logits
            )

            # ----------------------------
            # Store Results
            # ----------------------------

            results["image_ids"].extend(
                batch["image_id"]
            )

            results["hair_color_targets"].extend(
                batch["hair_color"].cpu().numpy()
            )

            results["hair_color_preds"].extend(
                hair_color_preds.cpu().numpy()
            )

            results["hair_texture_targets"].extend(
                batch["hair_texture"].cpu().numpy()
            )

            results["hair_texture_preds"].extend(
                hair_texture_preds.cpu().numpy()
            )

            results["binary_targets"].extend(
                batch["binary_labels"].cpu().numpy()
            )

            results["binary_logits"].extend(
                binary_logits.cpu().numpy()
            )

            results["binary_probs"].extend(
                binary_probs.cpu().numpy()
            )

# ------------------------------------------------------------------
# Convert everything to NumPy
# ------------------------------------------------------------------

results["hair_color_targets"] = np.asarray(results["hair_color_targets"])
results["hair_color_preds"] = np.asarray(results["hair_color_preds"])

results["hair_texture_targets"] = np.asarray(results["hair_texture_targets"])
results["hair_texture_preds"] = np.asarray(results["hair_texture_preds"])

results["binary_targets"] = np.asarray(results["binary_targets"])
results["binary_logits"] = np.asarray(results["binary_logits"])
results["binary_probs"] = np.asarray(results["binary_probs"])

# ------------------------------------------------------------------
# Valid Hair Texture Mask
# ------------------------------------------------------------------

results["valid_texture_mask"] = (
    results["hair_texture_targets"] != -1
)

# ------------------------------------------------------------------
# Inference Statistics
# ------------------------------------------------------------------

elapsed_time = time.time() - start_time

print("=" * 60)
print("Inference Completed")
print("=" * 60)

print(f"Total Images          : {len(results['image_ids'])}")
print(f"Inference Time (sec)  : {elapsed_time:.2f}")
print(f"Images / Second       : {len(results['image_ids']) / elapsed_time:.2f}")

print("\nPrediction Shapes")
print("-" * 60)

print(f"Hair Color     : {results['hair_color_preds'].shape}")
print(f"Hair Texture   : {results['hair_texture_preds'].shape}")
print(f"Binary Logits  : {results['binary_logits'].shape}")
print(f"Binary Probs   : {results['binary_probs'].shape}")

assert results["binary_probs"].shape[1] == 15, \
    "Binary output should contain exactly 15 attributes."

assert len(results["hair_color_targets"]) == len(results["hair_color_preds"])
assert len(results["hair_texture_targets"]) == len(results["hair_texture_preds"])
assert len(results["image_ids"]) == len(results["hair_color_targets"])

print("\nInference completed successfully.")


Running Inference:   0%|          | 0/634 [00:00<?, ?it/s]

KeyError: 'binary_labels'

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

# ============================================================
# Hair Color Evaluation
# ============================================================

print("=" * 70)
print("HAIR COLOR EVALUATION")
print("=" * 70)

y_true = results["hair_color_targets"]
y_pred = results["hair_color_preds"]

# ------------------------------------------------------------
# Overall Accuracy
# ------------------------------------------------------------

accuracy = accuracy_score(y_true, y_pred)

print(f"\nOverall Accuracy : {accuracy:.4f} ({accuracy*100:.2f}%)")

# ------------------------------------------------------------
# Classification Report
# ------------------------------------------------------------

print("\nClassification Report\n")

print(
    classification_report(
        y_true,
        y_pred,
        target_names=HAIR_COLOR_CLASSES,
        digits=4,
        zero_division=0
    )
)

# ------------------------------------------------------------
# Per-Class Accuracy
# ------------------------------------------------------------

cm = confusion_matrix(y_true, y_pred)

per_class_accuracy = cm.diagonal() / cm.sum(axis=1)

per_class_df = pd.DataFrame({
    "Hair Color": HAIR_COLOR_CLASSES,
    "Correct": cm.diagonal(),
    "Total": cm.sum(axis=1),
    "Accuracy": per_class_accuracy
})

per_class_df["Accuracy (%)"] = (
    per_class_df["Accuracy"] * 100
).round(2)

print("\nPer-Class Accuracy\n")

display(
    per_class_df.drop(columns="Accuracy")
)

# ------------------------------------------------------------
# Raw Confusion Matrix
# ------------------------------------------------------------

plt.figure(figsize=(8,6))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=HAIR_COLOR_CLASSES,
    yticklabels=HAIR_COLOR_CLASSES
)

plt.title("Hair Color Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("True")

plt.tight_layout()

plt.savefig(
    OUTPUT_DIR / "hair_color_confusion_matrix.png",
    dpi=300
)

plt.show()

# ------------------------------------------------------------
# Normalized Confusion Matrix
# ------------------------------------------------------------

cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

plt.figure(figsize=(8,6))

sns.heatmap(
    cm_norm,
    annot=True,
    fmt=".2f",
    cmap="Blues",
    xticklabels=HAIR_COLOR_CLASSES,
    yticklabels=HAIR_COLOR_CLASSES
)

plt.title("Normalized Hair Color Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("True")

plt.tight_layout()

plt.savefig(
    OUTPUT_DIR / "hair_color_confusion_matrix_normalized.png",
    dpi=300
)

plt.show()

# ------------------------------------------------------------
# Most Common Misclassifications
# ------------------------------------------------------------

misclassifications = []

for true_class in range(len(HAIR_COLOR_CLASSES)):
    for pred_class in range(len(HAIR_COLOR_CLASSES)):

        if true_class == pred_class:
            continue

        count = cm[true_class, pred_class]

        if count > 0:
            misclassifications.append({
                "True": HAIR_COLOR_CLASSES[true_class],
                "Predicted": HAIR_COLOR_CLASSES[pred_class],
                "Count": int(count)
            })

misclassification_df = (
    pd.DataFrame(misclassifications)
    .sort_values("Count", ascending=False)
    .reset_index(drop=True)
)

print("\nMost Common Misclassifications\n")

display(misclassification_df.head(10))